# Packages and ngspice setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

!sudo apt-get install ngspice libngspice0

# Create a symbolic link for libngspice.so if it doesn't exist
# This is often needed because PySpice looks for libngspice.so (without the version number)
!if [ ! -f /usr/lib/x86_64-linux-gnu/libngspice.so ]; then \
    sudo ln -s /usr/lib/x86_64-linux-gnu/libngspice.so.0 /usr/lib/x86_64-linux-gnu/libngspice.so; \
    echo "Created symlink /usr/lib/x86_64-linux-gnu/libngspice.so"; \
fi

# Set the environment variable for PySpice to find ngspice
import os
os.environ['PYSPICE_NGSPICE_PATH'] = '/usr/lib/x86_64-linux-gnu/libngspice.so.0'

!pip install PySpice

# Diagnostic: Find the actual path of libngspice.so
# Please share the output of this command after execution.
!dpkg -L ngspice
!dpkg -L libngspice0

# Further Diagnostics: Check file existence and dependencies
!ls -l /usr/lib/x86_64-linux-gnu/libngspice.so
!ldd /usr/lib/x86_64-linux-gnu/libngspice.so

import PySpice.Logging.Logging as Logging
import logging # Import the standard logging module
from PySpice.Spice.Netlist import Circuit
from PySpice.Unit import *
from PySpice.Spice.Parser import SpiceParser
import re
import torch
import scipy.signal as signal
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import concurrent.futures
import multiprocessing
import random
import glob

# Suppress specific PySpice warnings
logger = Logging.setup_logging()
logger.setLevel(logging.ERROR) # Use logging.ERROR instead of Logging.ERROR


# Netlist Processor (SPICE)

In [ ]:
class NetlistProcessor:
    """
    Motor de análisis de Netlists para el Kaspix Omni-Pipeline.
    Encargado de parsear, validar y extraer la configuración paramétrica
    de circuitos SPICE estándar.
    """

    def __init__(self, file_path):
        self.file_path = file_path
        self.content = ""
        self.params_raw = {}      # Diccionario original (texto, ej: '10k')
        self.params_float = {}    # Diccionario numérico (float, ej: 10000.0)
        self.io_config = {        # Metadatos de entrada/salida
            "input_source": None,
            "output_node": None,
            "ground_check": False
        }
        self.is_parsed = False

        # Carga inmediata
        self._load_file()

    def _load_file(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"[Kaspix Error] No se encuentra el archivo: {self.file_path}")

        with open(self.file_path, 'r', encoding='utf-8') as f:
            self.content = f.read()

    def _spice_to_float(self, value_str):
        """
        Convierte sufijos de ingeniería SPICE a float de Python.
        Soporta: T, G, Meg, k, m, u, n, p, f
        """
        value_str = value_str.lower().strip('{} ') # Limpiar llaves y espacios

        # Mapa de multiplicadores SPICE
        suffixes = {
            't': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3,
            'mil': 25.4e-6, 'm': 1e-3, 'u': 1e-6,
            'n': 1e-9, 'p': 1e-12, 'f': 1e-15
        }

        # Regex para separar número y sufijo
        # Busca un número (int o float) seguido opcionalmente de letras
        match = re.match(r'^([\d\.\-\+]+)([a-z]*)$', value_str)

        if not match:
            try:
                return float(value_str) # Intento directo
            except ValueError:
                return None # No es un número parseable (ej: una fórmula compleja)

        number, suffix = match.groups()
        multiplier = 1.0

        if suffix:
            # SPICE es 'greedy' con los sufijos, 'meg' es especial
            if suffix.startswith('meg'):
                multiplier = suffixes['meg']
            elif suffix[0] in suffixes:
                multiplier = suffixes[suffix[0]]

        return float(number) * multiplier

    def analyze(self):
        """
        Ejecuta el análisis léxico del netlist.
        """
        lines = self.content.splitlines()

        # Regex Compilados
        re_param = re.compile(r'\.param\s+(\w+)\s*=\s*(\{?[\w\.\-\+]+\}?)', re.IGNORECASE)
        re_source = re.compile(r'^\s*([vV]\w+)\s+(\w+)\s+(\w+)', re.IGNORECASE)
        re_save = re.compile(r'\.(?:save|print)\s+(?:v|tran)\s*\((.+?)\)', re.IGNORECASE)

        for line in lines:
            line = line.strip()
            if not line or line.startswith('*'): continue

            # 1. Detectar Parámetros (.param)
            pm = re_param.search(line)
            if pm:
                name, val_str = pm.groups()
                # Guardar raw
                self.params_raw[name] = val_str
                # Convertir a float
                val_float = self._spice_to_float(val_str)
                if val_float is not None:
                    self.params_float[name] = val_float

            # 2. Detectar Inputs (Heurística: Nombre contiene 'in' o 'src')
            sm = re_source.match(line)
            if sm:
                s_name, n_pos, n_neg = sm.groups()
                # Priorizamos fuentes que parezcan de señal
                if 'in' in s_name.lower() or 'src' in s_name.lower() or 'sig' in s_name.lower():
                    self.io_config["input_source"] = s_name

            # 3. Detectar Ground
            if ' 0 ' in line or line.endswith(' 0'):
                self.io_config["ground_check"] = True

            # 4. Detectar Output (.save)
            svm = re_save.search(line)
            if svm:
                # Extraer contenido dentro de paréntesis
                nodes = svm.group(1).replace(')', '').replace('(', '').split()
                if nodes:
                    self.io_config["output_node"] = nodes[0] # Tomamos el primero como principal

        # Fallback para Output si no hay .save
        if not self.io_config["output_node"]:
             if re.search(r'\b(out|output|salida)\b', self.content, re.IGNORECASE):
                 self.io_config["output_node"] = "out (Inferred)"
             else:
                 self.io_config["output_node"] = "UNKNOWN (Define .save V(node))"

        self.is_parsed = True
        return self.get_summary()

    def get_summary(self):
        if not self.is_parsed: return "No analizado."
        return {
            "file_path": self.file_path,
            "valid_spice": self.io_config["ground_check"],
            "knobs_detected": list(self.params_float.keys()),
            "knobs_values": self.params_float,
            "input_source": self.io_config["input_source"],
            "output_target": self.io_config["output_node"]
        }

In [ ]:
import subprocess
import numpy as np
import os

def run_power_simulation(vin, l_val, c_val, r_load):
    # Leemos el archivo base y reemplazamos los parámetros
    with open("LTC3417A_universal.cir", "r") as f:
        content = f.read()

    # Reemplazo dinámico de Knobs
    content = content.replace("{Vin_val}", f"{vin}V")
    content = content.replace("{Lval}", f"{l_val}uH")
    content = content.replace("{Cval}", f"{c_val}uF")
    content = content.replace("{Rload}", f"{r_load}")

    with open("sim_temp.cir", "w") as f:
        f.write(content)

    # Ejecutamos NGSpice en modo batch
    subprocess.run(["ngspice", "-b", "sim_temp.cir"], stdout=subprocess.DEVNULL)

    # Cargamos los resultados
    data = np.loadtxt("output.txt")
    return data[:, 0], data[:, 1], data[:, 3] # tiempo, v_out (target), v_in (input)

# Dataset Generation

In [ ]:
# ============================================================================
# 0. FUNCIONES AUXILIARES - REPRODUCIBILIDAD
# ============================================================================
def set_global_seed(seed=42):
    """
    Establece semillas aleatorias para reproducibilidad total.

    Args:
        seed: Semilla (int)
    """
    import random
    import numpy as np
    import torch

    # Python random
    random.seed(seed)

    # NumPy random
    np.random.seed(seed)

    # PyTorch random
    torch.manual_seed(seed)

    # Si usas CUDA (GPU)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # Para multi-GPU

        # Configuración para reproducibilidad CUDA (más lento pero determinista)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    print(f"✅ Kaspix Seed Locked: {seed}")

# ============================================================
# 1. KASPIX SIGNAL FACTORY (Con Dinámica de Amplitud)
# ============================================================
class KaspixSignalFactory:
    def __init__(self, fs, duration_sec):
        self.fs = fs
        self.duration = duration_sec
        self.n_samples = int(fs * duration_sec)
        self.time_axis = np.linspace(0, duration_sec, self.n_samples)

    def _apply_dynamic_gain(self, y):
        """
        Simula la variación de volumen de entrada (Dinámica).
        """
        target_peak = np.exp(np.random.uniform(np.log(0.05), np.log(1.5)))
        max_val = np.max(np.abs(y))
        if max_val > 1e-9:
            return (y / max_val) * target_peak
        return y

    def _apply_input_noise(self, signal_in):
        """
        [NUEVO] Inyecta un 'Piso de Ruido' realista.
        Simula interferencia eléctrica, térmica o de cables.
        """
        # 1. Decidir si esta muestra tendrá ruido (90% de las veces sí)
        if np.random.rand() > 0.9:
            return signal_in

        # 2. Generar Ruido Base (Blanco)
        noise = np.random.randn(len(signal_in))

        # 3. Determinar nivel de ruido (SNR)
        # Un 'Noise Floor' típico en audio va de -60dB (bueno) a -30dB (malo/vintage)
        # Calculamos la amplitud del ruido basada en una referencia fija (no relativa a la señal)
        # Esto simula que el ruido es constante independientemente de si tocas fuerte o suave.
        noise_level_db = np.random.uniform(-70, -30)
        noise_amplitude = 10 ** (noise_level_db / 20)

        # 4. Mezclar
        noisy_signal = signal_in + (noise * noise_amplitude)

        # Opcional: Recortar picos si se pasa de +/- 1.5V (Safety Clip)
        return np.clip(noisy_signal, -1.5, 1.5)

    # --- Generadores Primitivos ---
    def _pink_noise(self):
        uneven = self.n_samples % 2
        X = np.random.randn(self.n_samples // 2 + 1 + uneven) + 1j * np.random.randn(self.n_samples // 2 + 1 + uneven)
        S = np.sqrt(np.arange(len(X)) + 1.)
        y = (np.fft.irfft(X / S)).real
        if uneven: y = y[:-1]
        # ORDEN FÍSICO: Señal -> Ganancia -> Ruido de Cable
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _chirp_log(self):
        f_start = 20
        f_end = np.random.uniform(self.fs/4, self.fs/2 * 0.9)
        y = signal.chirp(self.time_axis, f0=f_start, f1=f_end, t1=self.duration, method='logarithmic')
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _step_sequence(self):
        num_steps = np.random.randint(3, 12)
        y = np.zeros_like(self.time_axis)
        indices = np.sort(np.random.choice(np.arange(self.n_samples), num_steps, replace=False))
        levels = np.random.uniform(-0.8, 0.8, num_steps)
        current_idx = 0
        for i, idx in enumerate(indices):
            y[current_idx:idx] = levels[i-1] if i > 0 else 0
            current_idx = idx
        y[current_idx:] = levels[-1]
        # Los steps ya tienen amplitud propia, solo añadimos ruido
        return self._apply_input_noise(y)

    def _multitone(self):
        num_tones = np.random.randint(3, 15)
        y = np.zeros_like(self.time_axis)
        for _ in range(num_tones):
            freq = np.random.uniform(20, self.fs/3)
            phase = np.random.uniform(0, 2*np.pi)
            y += np.sin(2 * np.pi * freq * self.time_axis + phase)
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _impulse_train(self):
        y = np.zeros_like(self.time_axis)
        num_impulses = np.random.randint(1, 5)
        indices = np.random.randint(0, self.n_samples, num_impulses)
        y[indices] = 1.0
        # Impulsos limpios suelen ser mejores para analisis, pero ruido leve no daña
        return self._apply_input_noise(y)

    def get_signal(self, recipe):
        keys = list(recipe.keys())
        probs = np.array(list(recipe.values()))
        probs /= probs.sum()
        choice = np.random.choice(keys, p=probs)

        sig = None
        name = "Unknown"

        if choice == 'chirp':
            sig, name = self._chirp_log(), "Chirp Log"
        elif choice == 'pink_noise':
            sig, name = self._pink_noise(), "Pink Noise"
        elif choice == 'step_sequence':
            sig, name = self._step_sequence(), "Step Seq"
        elif choice == 'multitone':
            sig, name = self._multitone(), "Multitone"
        elif choice == 'impulse':
            sig, name = self._impulse_train(), "Impulse"
        elif choice == 'sine':
            f = np.random.uniform(20, 1000)
            raw = np.sin(2*np.pi*f*self.time_axis)
            sig, name = self._apply_input_noise(self._apply_dynamic_gain(raw)), f"Sine {int(f)}Hz"
        elif choice == 'silence_decay':
             raw = np.zeros_like(self.time_axis)
             raw[:int(self.n_samples*0.1)] = np.random.randn(int(self.n_samples*0.1))
             # El silencio DEBE tener ruido de fondo para ser realista
             sig, name = self._apply_input_noise(self._apply_dynamic_gain(raw)), "Silence Decay"
        else:
            sig, name = self._pink_noise(), "Default (Pink)"

        return sig, name

# ============================================================
# 2. WORKER SIMULATOR (Con soporte para Dummy Knob)
# ============================================================
def _simulation_worker(task_payload):
    import numpy as np
    import re
    from PySpice.Spice.Parser import SpiceParser

    # --- SANITIZADOR DE NETLIST (Regex) ---
    def sanitize_netlist(content):
        """
        1. Elimina espacios dentro de llaves {} para arreglar fórmulas matemáticas.
           Ej: {R * 0.5} -> {R*0.5}
        2. Mueve .subckt al inicio (por seguridad, aunque usamos VCVS fix).
        """
        # Regex para eliminar espacios SOLO dentro de llaves
        content = re.sub(r'\{([^{}]*)\}', lambda m: '{' + m.group(1).replace(' ', '') + '}', content)

        # Reordenamiento de subcircuitos
        lines = content.splitlines()
        if not lines: return content
        title = lines[0]
        body = []
        subckts = []
        in_subckt = False
        buffer = []

        for line in lines[1:]:
            l_lower = line.strip().lower()
            if l_lower.startswith('.subckt'):
                in_subckt = True
                buffer.append(line)
            elif l_lower.startswith('.ends'):
                in_subckt = False
                buffer.append(line)
                subckts.extend(buffer)
                buffer = []
            elif in_subckt:
                buffer.append(line)
            else:
                body.append(line)
        return "\n".join([title] + subckts + body)
    # --------------------------------------

    try:
        # Desempaquetado
        file_path = task_payload['file_path']
        input_source = task_payload['input_source']
        output_target = task_payload['output_target']
        fs = float(task_payload['fs'])
        duration = float(task_payload['duration'])
        input_signal = task_payload['input_signal']
        knob_config = task_payload['knob_config']
        task_id = task_payload['id']

        # 1. LEER Y SANITIZAR
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_content = f.read()

        # APLICAR SANITIZACIÓN (Aquí se arregla el Notch Filter)
        clean_content = sanitize_netlist(raw_content)

        parser = SpiceParser(source=clean_content)
        circuit = parser.build_circuit()

        # --- FIX NUCLEAR: REEMPLAZO DE OPAMP X1 POR VCVS NATIVO ---
        # Detectamos subcircuitos X (OpAmps) y los cambiamos por primitivas E (VCVS)
        x_element_name = None
        for elm_name in list(circuit.element_names):
            if elm_name.upper().startswith('X'):
                x_element_name = elm_name
                break

        if x_element_name:
            x_component = circuit[x_element_name]
            # Asumimos orden standard: non_inv, inv, out
            if len(x_component.nodes) >= 3:
                node_non_inv = x_component.nodes[0]
                node_inv     = x_component.nodes[1]
                node_out     = x_component.nodes[2]

                circuit._elements.pop(x_element_name)

                # Inyectar VCVS (Fuente de voltaje controlada por voltaje)
                # Sintaxis interna: (name, node_plus, node_minus, ctl_plus, ctl_minus, gain)
                circuit.VCVS(
                    'OpAmp_Replacer',
                    node_out,
                    circuit.gnd,
                    node_non_inv,
                    node_inv,
                    voltage_gain=100000
                )

        # 2. INYECCIÓN DE FUENTE PWL
        actual_source_name = None
        for element in circuit.element_names:
            if element.upper() == input_source.upper():
                actual_source_name = element
                break

        if actual_source_name:
            original_source = circuit[actual_source_name]
            n_pos = str(original_source.nodes[0])
            n_neg = str(original_source.nodes[1])
            circuit._elements.pop(actual_source_name)

            time_axis = np.linspace(0, duration, len(input_signal))
            input_data = list(zip(
                [float(t) for t in time_axis],
                [float(v) for v in input_signal]
            ))
            circuit.PieceWiseLinearVoltageSource('Input_UES', n_pos, n_neg, values=input_data)
        else:
            return {"status": "error", "msg": f"Fuente '{input_source}' no hallada", "id": task_id}

        # 3. APLICAR KNOBS (Inyección y Sanitización)
        for name, val in knob_config.items():
            if name == 'dummy_param': continue
            val_f = float(val)

            # Inyección robusta en componentes o parámetros
            if name in circuit.element_names:
                obj = circuit[name]
                if hasattr(obj, 'resistance'): obj.resistance = val_f
                elif hasattr(obj, 'capacitance'): obj.capacitance = val_f
                elif hasattr(obj, 'inductance'): obj.inductance = val_f
                elif hasattr(obj, 'dc_value'): obj.dc_value = val_f
                elif hasattr(obj, 'voltage_gain'): obj.voltage_gain = val_f
            else:
                circuit.parameter(name, val_f)

        # 4. SIMULACIÓN
        simulator = circuit.simulator(temperature=25, nominal_temperature=25)
        simulator.options(reltol=0.01, method='gear', abstol='1e-10')

        try:
            step = float(1.0/fs)
            end_t = float(duration)
            analysis = simulator.transient(step_time=step, end_time=end_t)
        except Exception as e:
            return {"status": "error", "msg": f"NGSpice Crash: {str(e)}", "id": task_id}

        # 5. EXTRACCIÓN Y LIMPIEZA
        target_clean = output_target.upper().replace('V(', '').replace(')', '').strip()
        found_node = None

        # Búsqueda de nodo output
        for node_name in analysis.nodes:
            if str(node_name).upper() == target_clean:
                found_node = node_name
                break
        if found_node is None:
            for node_name in analysis.nodes:
                if str(node_name).upper() == 'OUT':
                    found_node = node_name
                    break
        # Último recurso: buscar substring 'OUT' o 'SALIDA'
        if found_node is None:
            for node_name in analysis.nodes:
                if 'OUT' in str(node_name).upper():
                    found_node = node_name
                    break

        if found_node is None:
            return {"status": "error", "msg": f"Salida '{target_clean}' no encontrada", "id": task_id}

        output_signal = np.array(analysis.nodes[found_node])

        # 6. CHEQUEOS DE SEGURIDAD
        if np.isnan(output_signal).any():
            return {"status": "error", "msg": "NaN Detectado", "id": task_id}
        if np.max(np.abs(output_signal)) > 200:
            return {"status": "error", "msg": "Explosión Numérica (>200V)", "id": task_id}

        if len(output_signal) != len(time_axis):
            output_signal = np.interp(time_axis, np.array(analysis.time), output_signal)

        return {
            "status": "ok",
            "id": task_id,
            "y": output_signal.astype(np.float32),
            "x_meta": {
                "knobs": np.array(list(knob_config.values()), dtype=np.float32),
                "knob_names": list(knob_config.keys())
            }
        }

    except Exception as e:
        return {"status": "error", "msg": f"Worker Logic Fail: {str(e)}", "id": task_payload.get('id', -1)}

#New worker
def _super_fast_worker(task_payload):
    import numpy as np
    import re
    from PySpice.Spice.Parser import SpiceParser
    from scipy.signal import correlate

    # Definición universal del OpAmp para que NGSpice no reclame
    IDEAL_OPAMP_SUB = """
.subckt ideal_opamp non_inv inv out
    E1 out 0 non_inv inv 100000
.ends
"""

    try:
        netlist_path = task_payload['file_path']
        knob_config = task_payload['knob_config']
        fs = task_payload['fs']
        duration = task_payload['duration']
        recipe = task_payload['recipe']
        seed = task_payload['seed']

        np.random.seed(seed)

        # 1. Generar señal local
        factory = KaspixSignalFactory(fs, duration)
        audio_in, sig_type = factory.get_signal(recipe)

        # 2. LEER E INYECTAR DEFINICIÓN (Solución al error "unknown subckt")
        with open(netlist_path, 'r') as f:
            lines = f.readlines()

        # Insertamos la definición justo después de la primera línea (título)
        raw_content = lines[0] + IDEAL_OPAMP_SUB + "".join(lines[1:])

        # 3. Construir Circuito
        parser = SpiceParser(source=raw_content)
        circuit = parser.build_circuit()

        # 4. Inyectar Entrada PWL y Knobs
        time_axis = np.linspace(0, duration, len(audio_in))
        input_data = list(zip(time_axis, audio_in))

        # Buscamos la fuente de entrada (usualmente Vin)
        circuit.PieceWiseLinearVoltageSource('Input_Signal', 'input', '0', values=input_data)

        for name, val in knob_config.items():
            if name in circuit.parameter_names:
                circuit.parameter(name, float(val))

        # 5. SIMULACIÓN ULTRA-RÁPIDA
        simulator = circuit.simulator(temperature=25)
        # Opciones críticas para que no tarde 7s por muestra
        simulator.options(reltol=0.01, method='gear', abstol='1e-10', itl4=100)

        analysis = simulator.transient(step_time=1.0/fs, end_time=duration)

        # Extraer salida (buscando 'output' o 'out')
        node_name = 'output' if 'output' in analysis.nodes else 'out'
        raw_target = np.array(analysis.nodes[node_name])

        if len(raw_target) != len(audio_in):
            raw_target = np.interp(time_axis, np.array(analysis.time), raw_target)

        # 6. ALINEACIÓN Y NORMALIZACIÓN (Garantiza ESR bajo)
        a_c = audio_in - np.mean(audio_in)
        t_c = raw_target - np.mean(raw_target)

        corr = correlate(t_c, a_c, mode='full')
        lag = np.arange(-len(a_c) + 1, len(a_c))[np.argmax(corr)]

        if lag > 0: a_aligned, t_aligned = a_c[:-lag], t_c[lag:]
        elif lag < 0: a_aligned, t_aligned = a_c[-lag:], t_c[:lag]
        else: a_aligned, t_aligned = a_c, t_c

        a_final = (a_aligned / (np.std(a_aligned) + 1e-8)).astype(np.float32)
        t_final = (t_aligned / (np.std(t_aligned) + 1e-8)).astype(np.float32)

        return {
            "status": "ok",
            "x": {"audio_in": a_final[:int(fs*duration)], "knobs": np.array(list(knob_config.values())), "type": sig_type},
            "y": t_final[:int(fs*duration)]
        }
    except Exception as e:
        return {"status": "error", "msg": str(e)}


# ============================================================
# 3. KASPIX GENERATOR (Con inyección de Dummy Knob)
# ============================================================
class KaspixParallelGenerator:
    def __init__(self, processor_result, fs=48000, duration_sec=0.2, recipe=None, seed=42):
        self.meta = processor_result
        self.fs = fs
        self.duration = duration_sec
        self.seed = seed
        self.factory = KaspixSignalFactory(fs, duration_sec)
        self.time_axis = self.factory.time_axis
        self.recipe = recipe if recipe else {"chirp": 1.0}

        if not self.meta['valid_spice']:
            raise ValueError("[Kaspix] Netlist inválido.")

        self.max_workers = multiprocessing.cpu_count()
        print(f"⚡ Kaspix Parallel Engine V6 (Dynamic Gain + Dummy Fix): {self.max_workers} núcleos | Seed: {self.seed}")

    def _sample_knobs(self, variation_pct=0.8):
        nominal = self.meta['knobs_values']

        # [MEJORA #3] Si no hay knobs detectados, inyectar uno falso
        if not nominal:
            # Retornamos un valor fijo (ej. 1.0) para mantener la estructura FiLM
            return {'dummy_param': 1.0}

        full_path = self.meta.get('file_path', '').lower()
        current_variation = 0.10 if 'notch' in full_path else variation_pct

        sampled = {}
        for name, val in nominal.items():
            delta = val * current_variation
            new_val = np.random.uniform(val - delta, val + delta)
            if new_val <= 0: new_val = val * 0.01
            sampled[name] = new_val
        return sampled

    def build_dataset(self, n_samples=100, save_path="kaspix_dataset.pt"):
        set_global_seed(self.seed)
        print(f"--- Preparando {n_samples} tareas deterministas ---")

        tasks = []
        pre_generated_signals = {}

        for i in range(n_samples):
            sig_in, sig_type = self.factory.get_signal(self.recipe)
            knobs = self._sample_knobs()

            pre_generated_signals[i] = {
                "audio_in": sig_in.astype(np.float32),
                "signal_type": sig_type
            }

            task = {
                "id": i,
                "file_path": self.meta['file_path'],
                "input_source": self.meta['input_source'],
                "output_target": self.meta['output_target'],
                "fs": self.fs,
                "duration": self.duration,
                "input_signal": sig_in,
                "knob_config": knobs
            }
            tasks.append(task)

        print(f"🚀 Procesando...")
        results_unsorted = []
        with concurrent.futures.ProcessPoolExecutor(max_workers=self.max_workers) as executor:
            results_unsorted = list(tqdm(executor.map(_simulation_worker, tasks), total=n_samples))

        print("📦 Ordenando y verificando consistencia...")
        results_sorted = sorted(results_unsorted, key=lambda x: x['id'])

        dataset_x = []
        dataset_y = []
        success_count = 0

        for res in results_sorted:
            idx = res['id']
            if res['status'] == 'ok':
                input_meta = pre_generated_signals[idx]
                worker_meta = res['x_meta']

                # 1. RECUPERAR SEÑALES
                audio_in = input_meta['audio_in']
                raw_target = res['y']

                # 2. LIMPIEZA Y ALINEACIÓN (Fix crítico para el Delay de 90 muestras)
                # Esta función debe estar definida en tu notebook
                audio_fix, target_fix, _ = clean_and_align(audio_in, raw_target)

                # 3. NORMALIZACIÓN Z-SCORE (Para que el ESR no explote)
                audio_norm = (audio_fix - np.mean(audio_fix)) / (np.std(audio_fix) + 1e-8)
                target_norm = (target_fix - np.mean(target_fix)) / (np.std(target_fix) + 1e-8)

                x_entry = {
                    "audio_in": audio_norm.astype(np.float32),
                    "signal_type": input_meta['signal_type'],
                    "knobs": worker_meta['knobs'],
                    "knob_names": worker_meta['knob_names']
                }
                dataset_x.append(x_entry)
                dataset_y.append(target_norm.astype(np.float32))
                success_count += 1
            else:
                if success_count == 0:
                    print(f"❌ Error en muestra {idx}: {res['msg']}")

        if success_count == 0:
            print("❌ FALLO TOTAL.")
            return [], []

        print(f"💾 Guardando {success_count}/{n_samples} muestras en {save_path}...")
        torch.save({
            "x": dataset_x,
            "y": dataset_y,
            "fs": self.fs,
            "meta": self.meta,
            "recipe": self.recipe,
            "seed": self.seed
        }, save_path)

        return dataset_x, dataset_y

In [ ]:
# Bloque extra

import concurrent.futures
import multiprocessing

def generate_super_fast_dataset(n_samples_per_circuit=20000):
    all_x, all_y = [], []
    circuits = ["active_bandpass_mfb.cir", "active_notch_twint.cir",
                "sallen_key_hpf.cir", "sallen_key_lpf.cir"]

    # Usar todos los núcleos de la CPU (En TPU/GPU Colab suelen ser 8 o 12)
    max_workers = multiprocessing.cpu_count()
    print(f"🚀 Iniciando Motor Paralelo Kaspix con {max_workers} núcleos...")

    for i, netlist in enumerate(circuits):
        cid = i + 4
        processor = NetlistProcessor(netlist)
        meta = processor.analyze()

        # 1. Preparar tareas para los núcleos
        tasks = []
        factory = KaspixSignalFactory(FS, DURATION)
        for s in range(n_samples_per_circuit):
            audio_in, _ = factory.get_signal({"pink_noise": 0.7, "chirp": 0.3})
            knobs = {k: v * np.random.uniform(0.1, 10.0) for k, v in meta['knobs_values'].items()}
            tasks.append({
                "id": s, "file_path": netlist, "input_source": meta['input_source'],
                "output_target": meta['output_target'], "fs": FS, "duration": DURATION,
                "input_signal": audio_in, "knob_config": knobs
            })

        # 2. EJECUCIÓN PARALELA (Aquí es donde recuperas la velocidad)
        print(f"📦 Simulando {netlist}...")
        with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
            # map reparte las tareas entre los núcleos automáticamente
            raw_results = list(tqdm(executor.map(_simulation_worker, tasks), total=n_samples_per_circuit))

        # 3. PROCESAMIENTO Y ALINEACIÓN (Post-simulación rápida)
        for res, tsk in zip(raw_results, tasks):
            if res['status'] == 'ok':
                # Alineación y limpieza de offset
                a_fix, t_fix, _ = clean_and_align(tsk['input_signal'], res['y'])

                # Normalización para que el ESR no explote
                a_norm = (a_fix - np.mean(a_fix)) / (np.std(a_fix) + 1e-8)
                target_norm = (t_fix - np.mean(t_fix)) / (np.std(t_fix) + 1e-8)

                all_x.append({
                    'audio_in': a_norm.astype(np.float32),
                    'knobs': res['x_meta']['knobs'],
                    'circuit_id': cid
                })
                all_y.append(target_norm.astype(np.float32))

    # Guardado Final
    torch.save({'x': all_x, 'y': all_y}, "/content/af_dataset.pt")
    print(" Dataset terminado en tiempo record.")

# Execution

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

if __name__ == "__main__":
    import os
    import torch
    import numpy as np

    # 1. CONFIGURACIÓN GLOBAL Y RUTAS
    # ------------------------------------------------------------
    DATA_DIR = "/content/drive/MyDrive/universal_filter_ngspice/datasets"
    OUTPUT_DATASET = "kaspix_ltc3417a_dataset.pt"
    os.makedirs(DATA_DIR, exist_ok=True)

    # Cantidad de simulaciones
    N_SIMULATIONS = 10000

    # 2. DEFINICIÓN DEL MODELO LTC3417A (MACROMODELO)

    NETLIST_TEMPLATE = """
    * Kaspix Power Project: Macromodelo LTC3417A
    .param Vin_val={vin}V
    .param Lval={l_val}uH
    .param Cval={c_val}uF
    .param Rload={r_load}
    .param fsw=4Meg

    V1 in 0 {{Vin_val}}
    Vclk clk 0 pulse(0 5 0 1n 1n 125n 250n)
    Bctrl ctrl 0 V = 2.5 * (1 - tanh(100 * (V(out) - 0.8)))

    S1 in sw ctrl 0 myswitch
    D1 0 sw mydiode
    L1 sw out {{Lval}}
    C1 out 0 {{Cval}}
    R1 out 0 {{Rload}}

    .model myswitch sw(vt=2.5 vh=0.5 ron=0.1 roff=1meg)
    .model mydiode d(is=1n n=1.1)

    .tran 50n 30u
    .control
      set wr_vecnames
      run
      wrdata output.txt v(out) v(in)
      quit
    .endc
    .end
    """

    all_data_x = []
    all_data_y = []

    print(f" INICIANDO KASPIX POWER-PIPELINE (LTC3417A)")

    # 3. BUCLE DE GENERACIÓN DE DATOS DE POTENCIA
    # ------------------------------------------------------------
    for i in range(N_SIMULATIONS):
        # Definimos los Knobs aleatorios dentro de rangos operativos reales
        v_in = np.round(np.random.uniform(2.5, 5.5), 2)
        l_val = np.round(np.random.uniform(0.47, 2.2), 2)
        c_val = np.round(np.random.uniform(4.7, 22.0), 2)
        r_load = np.round(np.random.uniform(2.0, 20.0), 2)

        if i % 100 == 0:
            print(f" Simulación {i}/{N_SIMULATIONS} | Vin: {v_in}V, L: {l_val}uH")
        # Inyectamos parámetros en el template
        current_netlist = NETLIST_TEMPLATE.format(
            vin=v_in, l_val=l_val, c_val=c_val, r_load=r_load
        )

        # Escribimos archivo temporal para NGSpice
        with open("temp_power.cir", "w") as f:
            f.write(current_netlist)

        # Ejecutamos simulación
        import subprocess
        subprocess.run(["ngspice", "-b", "temp_power.cir"], stdout=subprocess.DEVNULL)

        # Procesamos salida
        if os.path.exists("output.txt"):
            # Agregamos skiprows=1 para saltar los encabezados 'time', 'v(out)', etc.
            try:
                data = np.loadtxt("output.txt", skiprows=1)

                # Tiempo: data[:,0], Vout: data[:,1], Vin: data[:,3]
                sample_x = {
                    'audio_in': torch.FloatTensor(data[:, 3]), # Estímulo (Vin)
                    'knobs': torch.FloatTensor([v_in, l_val, c_val, r_load]),
                    'sensor_id': torch.tensor(0),
                    'netlist_origin': "LTC3417A_Behavioral"
                }
                sample_y = torch.FloatTensor(data[:, 1]) # Target (Vout)

                all_data_x.append(sample_x)
                all_data_y.append(sample_y)
            except Exception as e:
                print(f" Error cargando simulación {i}: {e}")

    # 4. GUARDADO CONSOLIDADO
    # ------------------------------------------------------------
    save_path = os.path.join(DATA_DIR, OUTPUT_DATASET)
    print(f"\n[Final] Consolidando {len(all_data_x)} simulaciones de potencia...")

    torch.save({
        'x': all_data_x,
        'y': all_data_y,
        'metadata': {
            'component': 'LTC3417A',
            'knob_map': ['Vin', 'L', 'C', 'Rload'],
            'n_simulations': len(all_data_x),
            'target_v': 0.8
        }
    }, save_path)

    print(f" Dataset de potencia guardado en: {save_path}")

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# 1. Cargar el dataset generado
dataset_path = "/content/drive/MyDrive/universal_filter_ngspice/datasets/kaspix_ltc3417a_dataset.pt"
checkpoint = torch.load(dataset_path)

# 2. Información General
print(" INFORMACIÓN DEL DATASET")
print(f"Total de simulaciones: {checkpoint['metadata']['n_simulations']}")
target_v = checkpoint['metadata'].get('target_v', 0.8)

# 3. VISUALIZACIÓN DE EJEMPLOS
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Tomamos 3 ejemplos aleatorios
indices = np.random.choice(len(checkpoint['x']), 3, replace=False)
colors = ['#1f77b4', '#2ca02c', '#d62728']

for idx, color in zip(indices, colors):
    v_out = checkpoint['y'][idx].numpy()
    knobs = checkpoint['x'][idx]['knobs']

    # LA CORRECCIÓN: Calcular 't' para CADA muestra
    # Usamos 30 (que es el tiempo final de la simulación .tran 30u)
    t_muestreo = np.linspace(0, 30, len(v_out))

    label = f"Vin={knobs[0]:.2f}V, L={knobs[1]:.2f}uH, C={knobs[2]:.2f}uF"

    # Gráfico 1: Transitorio Completo (Startup)
    ax1.plot(t_muestreo, v_out, color=color, alpha=0.8, label=label)

    # Gráfico 2: ZOOM al Rizado (Ripple) - últimos 100 puntos
    # Solo graficamos si hay suficientes puntos
    if len(v_out) > 100:
        ax2.plot(t_muestreo[-100:], v_out[-100:], color=color, alpha=0.9, lw=2)

# Estética del gráfico principal
ax1.set_title("Respuesta Transitoria del LTC3417A (Startup)", fontsize=14)
ax1.set_ylabel("Voltaje de Salida (V)", fontsize=12)
ax1.axhline(target_v, color='black', linestyle='--', alpha=0.5, label=f"Referencia {target_v}V")
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)

# Estética del Zoom (Ripple)
ax2.set_title("Zoom del Rizado (Ripple) Final", fontsize=14)
ax2.set_ylabel("Voltaje (V)", fontsize=12)
ax2.set_xlabel("Tiempo (µs)", fontsize=12)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()